# Projet de cours initial : classification en/hors agglomération

Archive des quatorze cellules de code du projet M1 initial. Les sorties historiques ont été retirées ; leurs scores ne sont pas validés par le protocole corrigé. Cette archive conserve notamment le découpage par ligne et les traitements d’origine, qui comportent des limites documentées.

L’analyse révisée, reproductible, se trouve dans [02_methodologie.ipynb](02_methodologie.ipynb) et dans `src/`. Voir [la distinction entre projet initial et extension](../docs/original-project.md).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
caract = pd.read_csv("../data/Caract_2024.csv", sep=";")
lieux = pd.read_csv("../data/Lieux_2024.csv", sep=";") #le séparateur=";"

df = caract.merge(lieux, on="Num_Acc", how="left") #innerjoin avec num accident
#"Caractéristiques" = le fichier de gauche gardé

df.shape, df.head()

In [ ]:
df_small = df.copy() #garder le fichier original

df = df.replace(-1, np.nan) #remplacer les -1 par des NaN

df_small["target"] = (df_small["agg"] == 2).astype(int) #créer la Target

df_small["target"].value_counts(), df_small["target"].value_counts(normalize=True) #cf. la répartition

In [ ]:
features = [
    "lum", "atm", "col",
    "jour", "mois", "hrmn",
    "catr", "circ", "nbv",
    "surf", "plan", "vma",
    "infra", "situ"
]
#liste manuellement saisi

features = [c for c in features if c in df_small.columns]
#on garde seulement celles qui existent vraiment

X = df_small[features].copy()
y = df_small["target"].copy()

X.shape, y.shape

In [ ]:
if "vma" in X.columns:
    X["vma"] = pd.to_numeric(X["vma"], errors="coerce")
#force la conversion en nombres: transforme tout ce qui n'est pas un chiffre 
#(ex: "50 km/h", "Inconnu", erreurs de frappe) en NaN (vide)

X.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
y.value_counts().plot(kind="bar")
plt.title("Répartition de la cible : 1=en agglomération, 0=hors agglomération")
plt.xlabel("Classe")
plt.ylabel("Nombre d'accidents")
plt.show()

In [ ]:
tab = pd.crosstab(df_small["lum"], df_small["target"], normalize="index")
tab
pd.crosstab(df_small["lum"], df_small["target"]).plot(kind="bar", stacked=True)
#Barplot: la proportion d'accidents selon la luminosité
plt.title("Cible selon la luminosité (lum)")
plt.xlabel("lum")
plt.ylabel("Nombre d'accidents")
plt.show()

In [ ]:
if "vma" in df_small.columns:
    df_small["vma_num"] = pd.to_numeric(df_small["vma"], errors="coerce")
    #transforme les NON chiffre en NaN
    df_small.boxplot(column="vma_num", by="target")
    #Boxplot: Montre la distribution de la vitesse pour la ville (1) et la campagne (0)
    
    plt.title("Vitesse max (vma) selon la cible")
    plt.suptitle("")
    plt.xlabel("target")
    plt.ylabel("vma")
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
) 
#coupe les données: 80% pour l'entraînement, 20% pour le test
#67%=ville: stratify oblige le découpage à mettre exactement 67% de ville dans le Train 
#et 67% de ville dans le Test
X_train.shape, X_test.shape

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if X[c].dtype != "object"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")), 
            #val.numérique: remplit les (NaN) par la Médiane
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")), 
            #val.catégoriel: remplit les trous par la valeur la plus fréquente
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
            #(OneHot)= transforme le texte en colonnes binaires 
            #handle_unknown: on met des 0 partout pr nouvelle catégorie inconnue
        ]), cat_cols),
    ],
    remainder="drop"
)

cat_cols, num_cols

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

log_reg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=300, class_weight="balanced"))
])
#classe Campagne=minoritaire, le modèle va l'ignorer 
#class_weight="balanced": les erreurs sur la campagne comptent double: pénalité globale

log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]

print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

RocCurveDisplay.from_estimator(log_reg, X_test, y_test)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
        #pénalité recalculée pour chaque petit arbre
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

RocCurveDisplay.from_estimator(rf, X_test, y_test)
plt.show()

In [ ]:
# entraîner preprocess seul pour récupérer les noms de colonnes transformées
preprocess.fit(X_train)

# noms des colonnes
num_names = num_cols

cat_encoder = preprocess.named_transformers_["cat"].named_steps["onehot"]
cat_names = list(cat_encoder.get_feature_names_out(cat_cols)) if len(cat_cols) else []

feature_names = num_names + cat_names

importances = rf.named_steps["model"].feature_importances_
imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
#les variables sont utilisées le plus souvent en haut des arbres pour bien séparer les classes
#la Vitesse (vma) arrive largement en tête, suivie du Type de route (catr)

imp.head(20)

In [ ]:
imp.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 variables importantes (Random Forest)")
plt.xlabel("Importance")
plt.show()